In [1]:
import os
import base64
import subprocess
import time
import warnings
from getpass import getpass

from google.genai import types
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

# --- NEW IMPORTS TO TRIGGER THE BUG ---
from pydantic import SecretStr, ValidationError
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.config.settings import EmbeddingConfig, EmbeddingProvider, Neo4jConfig
from neo4j_agent_memory.integrations.google_adk import Neo4jMemoryService

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
os.environ["GOOGLE_API_KEY"] = getpass("Enter Google API Key: ")
os.environ["GEMINI_MODEL"] = "gemini-flash-latest"

os.environ["NEO4J_URI"] = "neo4j+s://demo.neo4jlabs.com"
os.environ["NEO4J_DATABASE"] = "companies"

# Encode credentials for HTTP Header
credentials = base64.b64encode(b"companies:companies").decode()

In [3]:
PORT = "8443"

# Start background server
process = subprocess.Popen(
    [
        "neo4j-mcp-server",
        "--neo4j-uri", os.environ["NEO4J_URI"],
        "--neo4j-transport-mode", "http",
        "--neo4j-http-port", PORT,
        "--neo4j-database", os.environ["NEO4J_DATABASE"],
    ],
    stdout=subprocess.PIPE, 
    stderr=subprocess.PIPE, 
    text=True
)

time.sleep(3) # Warm-up time

# Check if the process started
poll = process.poll()
if poll is not None:
    stdout, stderr = process.communicate()
    print(f"Server failed to start (Exit Code: {poll})")
else:
    print(f"Server started successfully on port {PORT}")

Server started successfully on port 8443


In [4]:
# Defining the tool with HTTP connection parameters
mcp_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"http://localhost:{PORT}/mcp",
        headers={"Authorization": f"Basic {credentials}"}
        ),
)

async def create_agent(model, name, prompt, tools_list):
    return LlmAgent(
        model=model,
        name=name,
        instruction=prompt,
        tools=tools_list
    )

system_prompt = """
You are a graph database assistant. Your job is to answer user questions by querying Neo4j.
Always run 'get-schema' first if you are unfamiliar with the graph structure.
"""

mcp_agent = await create_agent(
    model=os.environ["GEMINI_MODEL"], 
    name="neo4j_explorer", 
    prompt=system_prompt, 
    tools_list=[mcp_tools]
)

In [5]:
async def ask_graph(agent, query):
    session_service = InMemorySessionService()
    artifacts_service = InMemoryArtifactService()

    print("\n[Setup] Initializing Neo4jMemoryService...")
    settings = MemorySettings(
        neo4j=Neo4jConfig(
            uri="neo4j+s://7f9900bf.databases.neo4j.io",
            username="neo4j",
            password="xyhqxri00HTV7sm4J3W0QVmUkewcqJBnNmmp8uvOrEk",
            database="neo4j"
        ),
        embedding=EmbeddingConfig(
            provider=EmbeddingProvider.VERTEX_AI,
            model="text-embedding-004",
            project_id="gcp-neo4j-agent-integr-14f4", 
            location="us-central1",
        ),
    )
    memory_client = MemoryClient(settings)
    print("[Setup] Neo4jMemoryService initialized successfully.")
    await memory_client.connect()
    print("[Setup] MemoryClient connected successfully.")
    neo4j_memory_service = Neo4jMemoryService(
        memory_client=memory_client,
        user_id="user_1",
    )

    session = await session_service.create_session(state={}, app_name="neo4j_explorer", user_id="user_1")
    content = types.Content(role='user', parts=[types.Part(text=query)])

    print("[Setup] Injecting Neo4jMemoryService into ADK Runner...")
    runner = Runner(
        app_name="neo4j_explorer",
        agent=agent,
        artifact_service=artifacts_service,
        session_service=session_service,
        memory_service=neo4j_memory_service
    )

    print(f"\nProcessing: {query}")
    try:
        events_async = runner.run_async(
            session_id=session.id, 
            user_id=session.user_id, 
            new_message=content
        )

        async for event in events_async:
            if hasattr(event, 'content') and event.content:
                for part in event.content.parts:
                    if part.text:
                        print(f"Result: {part.text}")
                        
        print("\n[Debug] Execution finished. Attempting to sync memory...")

        # 1. Fetch the FRESH session from the service (contains the new history)
        fresh_session = await session_service.get_session(session_id=session.id,app_name="neo4j_explorer", user_id=session.user_id)

        # 2. Manually trigger the save
        try:
            await neo4j_memory_service.add_session_to_memory(fresh_session)
            print("[Success] add_session_to_memory called successfully.")
        except Exception as e:
            print(f"[Error] Manual save failed: {e}")
            
    except ValidationError as e:
        print("\n" + "="*50)
        print("🚨 BUG REPRODUCED: Pydantic ValidationError Caught 🚨")
        print("="*50)
        print(f"Error Details:\n{e}")
        print("="*50)
    except Exception as e:
        print(f"Unexpected Error: {e}")

# Example Query
await ask_graph(mcp_agent, "Who is CEO of Neo4j?")


[Setup] Initializing Neo4jMemoryService...
[Setup] Neo4jMemoryService initialized successfully.
[Setup] MemoryClient connected successfully.
[Neo4jMemoryService] Initialized with user_id=user_1, include_entities=True, include_preferences=True, extract_on_store=True
[Setup] Injecting Neo4jMemoryService into ADK Runner...

Processing: Who is CEO of Neo4j?


/usr/local/python/3.12.1/lib/python3.12/contextlib.py:105: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)
/usr/local/python/3.12.1/lib/python3.12/site-packages/google/adk/tools/mcp_tool/mcp_toolset.py:313: DeprecationWarning: MCPTool class is deprecated, use `McpTool` instead.
  mcp_tool = MCPTool(


Result: The CEO of Neo4j is Emil Eifrem.

[Debug] Execution finished. Attempting to sync memory...


/usr/local/python/3.12.1/lib/python3.12/site-packages/pydantic/main.py:250: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
Stage 'SpacyEntityExtractor' failed: spaCy is required for SpacyEntityExtractor. Install with: pip install spacy && python -m spacy download en_core_web_sm
Stage 'GLiNEREntityExtractor' failed: GLiNER is required for GLiNEREntityExtractor. Install with: pip install gliner
Stage 'LLMEntityExtractor' failed: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
Error storing message: 'neo4j_explorer' is not a valid MessageRole
Error storing message: 'neo4j_explorer' is not a valid MessageRole
Error storing message: 'neo4j_explorer' is not a valid Mes

[Success] add_session_to_memory called successfully.
